In [20]:
!pip install sklearn_crfsuite

In [21]:
import pandas as pd
import ast
import re

# 1. Cargar tu dataset
df = pd.read_csv("dataset_tecnico_preprocesado.csv")

def generar_bio_desde_palabras_clave(row):
    texto = str(row['texto'])
    palabras = row['palabras_clave']

    # Si palabras_clave viene formateado como string (ej: "['GCP', 'Adobe']"), lo convertimos a lista
    if isinstance(palabras, str):
        try:
            palabras = ast.literal_eval(palabras)
        except (ValueError, SyntaxError):
            palabras = []

    if not isinstance(palabras, list):
        palabras = []

    # Inicializamos todas las etiquetas como 'O' (Outside)
    etiquetas = ['O'] * len(texto)

    # Buscamos la posición de cada palabra clave en el texto
    for kw in palabras:
        kw_str = str(kw).strip()
        if not kw_str:
            continue

        pattern = re.compile(re.escape(kw_str), re.IGNORECASE)
        for match in pattern.finditer(texto):
            start, end = match.span()

            # Asignamos 'B' al primer carácter de la palabra clave
            etiquetas[start] = 'B'
            # Asignamos 'I' a los caracteres restantes de la palabra clave
            for i in range(start + 1, end):
                etiquetas[i] = 'I'

    return ",".join(etiquetas)

print("Generando columna 'etiquetas_bio' a partir de 'palabras_clave'...")
df['etiquetas_bio'] = df.apply(generar_bio_desde_palabras_clave, axis=1)

# Guardamos el CSV actualizado con la nueva columna
df.to_csv("dataset_tecnico_preprocesado.csv", index=False)
print("¡Columna 'etiquetas_bio' generada y guardada exitosamente!")

⏳ Generando columna 'etiquetas_bio' a partir de 'palabras_clave'...
¡Columna 'etiquetas_bio' generada y guardada exitosamente!


In [22]:
#Entrenamiento y Evaluación del Modelo CRF
import pandas as pd
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split

# 1. Cargar el dataframe procesado
df = pd.read_csv("dataset_tecnico_preprocesado.csv")

# Eliminar nulos remanentes
df = df.dropna(subset=['texto', 'etiquetas_bio']).copy()

# 2. Función para extraer características de cada carácter (Feature Engineering)
def extraer_caracteristicas_caracter(texto, i):
    char = texto[i]
    features = {
        'bias': 1.0,
        'char': char,
        'is_alnum': char.isalnum(),
        'is_alpha': char.isalpha(),
        'is_digit': char.isdigit(),
        'is_upper': char.isupper(),
        'is_space': char.isspace(),
        'is_punct': not char.isalnum() and not char.isspace(),
    }

    # Contexto izquierdo (carácter anterior)
    if i > 0:
        char_prev = texto[i-1]
        features.update({
            '-1:char': char_prev,
            '-1:is_space': char_prev.isspace(),
            '-1:is_punct': not char_prev.isalnum() and not char_prev.isspace(),
        })
    else:
        features['BOS'] = True

    # Contexto derecho (carácter siguiente)
    if i < len(texto) - 1:
        char_next = texto[i+1]
        features.update({
            '+1:char': char_next,
            '+1:is_space': char_next.isspace(),
            '+1:is_punct': not char_next.isalnum() and not char_next.isspace(),
        })
    else:
        features['EOS'] = True

    return features

def texto_a_features(texto):
    return [extraer_caracteristicas_caracter(texto, i) for i in range(len(texto))]

# 3. Mapeo de X e Y con verificación estricta de longitud (|X| == |Y|)
print("Extrayendo características a nivel de caracteres para el CRF...")

X = []
y = []
filas_descartadas = 0

for texto_raw, etiquetas_raw in zip(df['texto'], df['etiquetas_bio']):
    texto = str(texto_raw)

    if isinstance(etiquetas_raw, str):
        etiquetas = [e.strip(" '\"[]") for e in etiquetas_raw.split(',')]
    elif isinstance(etiquetas_raw, list):
        etiquetas = etiquetas_raw
    else:
        continue

    features = texto_a_features(texto)

    # Validación de alineación
    if len(features) == len(etiquetas):
        X.append(features)
        y.append(etiquetas)
    else:
        filas_descartadas += 1

print(f" Extracción completada. Muestras listas: {len(X)}")
if filas_descartadas > 0:
    print(f" Filas descartadas por desalineación: {filas_descartadas}")

# 4. Partición de Datos (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f" Entrenamiento: {len(X_train)} textos | Evaluación: {len(X_test)} textos")

# 5. Entrenamiento del Modelo CRF
modelo_crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # Regularización L1
    c2=0.1,  # Regularización L2
    max_iterations=100,
    all_possible_transitions=True
)

print("\nEntrenando el modelo CRF...")
modelo_crf.fit(X_train, y_train)
print("¡Entrenamiento completado!")

# 6. Reporte de Evaluación
y_pred = modelo_crf.predict(X_test)
labels = list(modelo_crf.classes_)

print("\n === REPORTE DE CLASIFICACIÓN DEL CRF (MÉTRICAS BIO) ===")
print(metrics.flat_classification_report(
    y_test, y_pred, labels=labels, digits=3
))

⏳ Extrayendo características a nivel de caracteres para el CRF...
 Extracción completada. Muestras listas: 5000
 Entrenamiento: 4000 textos | Evaluación: 1000 textos

⏳ Entrenando el modelo CRF...
¡Entrenamiento completado!

 === REPORTE DE CLASIFICACIÓN DEL CRF (MÉTRICAS BIO) ===
              precision    recall  f1-score   support

           O      0.900     0.979     0.938     90360
           B      0.697     0.342     0.459      1353
           I      0.595     0.230     0.331     11725

    accuracy                          0.885    103438
   macro avg      0.731     0.517     0.576    103438
weighted avg      0.863     0.885     0.863    103438

